# CartoonSet Preprocessing -- Captions and CLIP Fine-Tuning

Prepares the CartoonSet dataset for text-conditioned OT-GAN training
(Section 6 of the thesis): loads the attribute CSVs into a DataFrame,
generates a randomized natural-language caption per image, fine-tunes CLIP
on the (image, caption) pairs, and pre-computes an embedding for every
image so GAN training never needs a CLIP forward pass.

Reusable code lives in `src/`: `captions.py` (`generate_clip_caption`),
`data.py` (`CartoonCLIPDataset`), `training.py` (`finetune_clip` and its
helpers), `metrics.py` (`precompute_embeddings`, `check_gap`,
`audit_all_attribute_gaps`), and `plotting.py` (`plot_grid_per_class`,
`plot_clip_clusters`).

In [ ]:
import os
import sys
import glob
import json
import random
import tarfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.abspath('../src'))  # make src/ importable

from captions import generate_clip_caption
from data import CartoonCLIPDataset
from training import finetune_clip
from metrics import precompute_text_embeddings, check_gap, audit_all_attribute_gaps
from plotting import plot_grid_per_class, plot_clip_clusters

# --- Paths: update these for your environment ---
TGZ_PATH = '/kaggle/input/datasets/tinasikhbse/cartoons-100k/cartoonset100k.tgz'
DATA_ROOT = '/kaggle/working/'
IMAGE_DIR = os.path.join(DATA_ROOT, 'cartoonset100k')

device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print(f"Using device: {device}")


## 1. Load the dataset

Extract the CartoonSet tarball and build a DataFrame from the per-image attribute CSVs.

In [ ]:
print("Extracting...")
with tarfile.open(TGZ_PATH, 'r:gz') as tar:
    tar.extractall(DATA_ROOT)
print("Done.")

In [ ]:
records = []
for subfolder in sorted(os.listdir(os.path.join(DATA_ROOT, "cartoonset100k"))):
    subfolder_path = os.path.join(DATA_ROOT, "cartoonset100k", subfolder)
    if not os.path.isdir(subfolder_path):
        continue
    for csv_path in sorted(glob.glob(os.path.join(subfolder_path, "cs*.csv"))):
        img_id = os.path.splitext(os.path.basename(csv_path))[0]
        img_path = os.path.join(subfolder_path, img_id + ".png")
        df_csv = pd.read_csv(csv_path, header=None, names=["feature", "value", "n_categories"])
        row = {"id": img_id, "img_path": img_path}
        for _, r in df_csv.iterrows():
            row[r["feature"]] = int(r["value"])
        records.append(row)

df = pd.DataFrame(records)
print(f"Loaded {len(df)} images, {len(df.columns)-2} features")
df.head(1)

## 2. Inspect attribute values

Quick visual sanity check: one random example image per distinct value, for
every attribute. Confirms the CSV-encoded integer IDs actually correspond to
the visual differences we expect before writing the caption-generation
logic below.


In [ ]:
# df = pd.read_csv('/kaggle/working/cartoon_attr.csv') 

# Define columns to evaluate sequentially
feature_cols = [
    'eye_angle', 'eye_lashes', 'eye_lid', 'chin_length', 'eyebrow_weight', 
    'eyebrow_shape', 'eyebrow_thickness', 'face_shape', 'facial_hair', 'hair', 
    'eye_color', 'face_color', 'hair_color', 'glasses', 'glasses_color', 
    'eye_slant', 'eyebrow_width', 'eye_eyebrow_distance'
]

# Base directory where cartoonset PNG images are stored
# IMAGE_DIR is set in the setup cell above

# 2. SEQUENTIAL DATA PARSING & PLOTTING (RANDOM ONLY, NO DISPLAY CAPPING)
for feature in feature_cols:
    if feature not in df.columns:
        continue
        
    # Find all distinct value labels existing in this dataset chunk
    distinct_labels = sorted(df[feature].unique())
    num_labels = len(distinct_labels)
    
    print(f"\n========================================================")
    print(f"PROCESSING FEATURE: '{feature}' | Visualizing All {num_labels} Distinct Values")
    print(f"========================================================")
    
    # Grid config: Single row layout, dynamic width matching the total unique labels exactly
    fig, axes = plt.subplots(1, num_labels, figsize=(num_labels * 2.5, 3))
    
    # If the feature has only 1 distinct label, matplotlib returns a single axis object instead of an array
    if num_labels == 1:
        axes = [axes]
        
    for idx, label_val in enumerate(distinct_labels):
        # Filter all matching rows for this distinct attribute value
        matching_rows = df[df[feature] == label_val]
        
        # Select exactly one random representative sample row
        random_match = matching_rows.sample(n=1).iloc[0]
        
        # Build image file path safely (stripping duplicate base directory formatting if present)
        clean_img_path = str(random_match['img_path']).replace('/kaggle/working/cartoonset100k/', '')
        img_path = os.path.join(IMAGE_DIR, clean_img_path)
        
        # Render the random representative sample image onto the subplot
        try:
            img = Image.open(img_path)
            axes[idx].imshow(img)
        except Exception:
            axes[idx].text(0.5, 0.5, "Image\nMissing", ha='center', va='center', color='red')
            
        axes[idx].set_title(f"Label Val: {label_val}", fontsize=10, fontweight='bold')
        axes[idx].axis('off')
        
    fig.suptitle(f"Random Continuous Distribution for Attribute: [{feature}]", fontsize=14, fontweight='bold', y=1.08)
    plt.tight_layout()
    plt.show()

## 3. Inspect attribute values under fixed constraints

Same idea as above, but constraining one or two *other* attributes so the
comparison isolates a single feature (e.g. eyebrow shape at fixed eyebrow
thickness) -- otherwise a random sample conflates several attributes
changing at once. `plot_grid_per_class` (in `src/plotting.py`) implements
this.


In [ ]:
# =====================================================================
# 1. Eyebrow Weight (3 examples per class, without glasses)
# =====================================================================
print("\n[1/7] Processing: Eyebrow Weight...")
mask_eb_weight = df['glasses'] == 11
plot_grid_per_class(df, 'eyebrow_weight', mask_eb_weight, num_samples_per_class=3, title_prefix="Without Glasses")


# =====================================================================
# 2. Eyebrow Shape (3 examples per class, with eyebrow_thickness = 3)
# =====================================================================
print("\n[2/7] Processing: Eyebrow Shape...")
mask_eb_shape = df['eyebrow_thickness'] == 3
plot_grid_per_class(df, 'eyebrow_shape', mask_eb_shape, num_samples_per_class=3, title_prefix="Constraint: Eyebrow Thickness = 3")


# =====================================================================
# 3. Eyebrow Thickness (3 examples per class, without glasses)
# =====================================================================
print("\n[3/7] Processing: Eyebrow Thickness...")
mask_eb_thick = df['glasses'] == 11
plot_grid_per_class(df, 'eyebrow_thickness', mask_eb_thick, num_samples_per_class=3, title_prefix="Without Glasses")


# =====================================================================
# 4. Face Shape (3 examples per class, with facial_hair = 14)
# =====================================================================
print("\n[4/7] Processing: Face Shape...")
mask_face_shape = df['facial_hair'] == 14
plot_grid_per_class(df, 'face_shape', mask_face_shape, num_samples_per_class=3, title_prefix="Constraint: No Facial Hair")


# =====================================================================
# 5. Hair (111 distinct values arranged into exactly 10 rows)
# =====================================================================
print("\n[5/7] Processing: Complete Hair Grid (111 styles in 10 rows)...")
distinct_hair = sorted(df['hair'].unique())
num_hair_classes = len(distinct_hair)

if num_hair_classes > 0:
    num_rows = 10
    # Dynamically find the required columns to fit 111 classes in 10 rows (~12 columns)
    num_cols = int(np.ceil(num_hair_classes / num_rows)) 
    
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 1.8, num_rows * 1.8))
    axes_flat = axes.flatten()
    
    for idx, hair_val in enumerate(distinct_hair):
        ax = axes_flat[idx]
        hair_rows = df[df['hair'] == hair_val]
        
        if len(hair_rows) > 0:
            random_sample = hair_rows.sample(n=1).iloc[0]
            clean_img_path = str(random_sample['img_path']).replace('/kaggle/working/cartoonset100k/', '')
            img_path = os.path.join(IMAGE_DIR, clean_img_path)
            
            try:
                img = Image.open(img_path)
                ax.imshow(img)
            except Exception:
                ax.text(0.5, 0.5, "Missing", ha='center', va='center', color='red', fontsize=8)
        else:
            ax.text(0.5, 0.5, "N/A", ha='center', va='center', color='gray', fontsize=8)
            
        ax.set_title(f"ID: {hair_val}", fontsize=8, fontweight='bold')
        ax.axis('off')
        
    # Blind fold/turn off remaining empty subplots at the bottom edge of the grid
    for idx in range(num_hair_classes, len(axes_flat)):
        axes_flat[idx].axis('off')
        
    fig.suptitle("Complete Uncapped Hair Styles Grid (10 Rows Layout)", fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()


# =====================================================================
# 6. Eye Color (3 examples per class, with glasses = 11)
# =====================================================================
print("\n[6/7] Processing: Eye Color...")
mask_eye_color = df['glasses'] == 11
plot_grid_per_class(df, 'eye_color', mask_eye_color, num_samples_per_class=3, title_prefix="Without Glasses")


# =====================================================================
# 7. Glasses Color (3 examples per class, with glasses = 1)
# =====================================================================
print("\n[7/7] Processing: Glasses Color...")
mask_glasses_color = df['glasses'] == 1
plot_grid_per_class(df, 'glasses_color', mask_glasses_color, num_samples_per_class=3, title_prefix="Constraint: Glasses Type = 1")

# =====================================================================
# 8. Facial Hair (3 examples per class, with glasses = 1)
# =====================================================================
print("\n[7/7] Processing: Facial Hair...")
plot_grid_per_class(df, 'facial_hair', mask_glasses_color, num_samples_per_class=3, title_prefix="Constraint: Glasses Type = 1")

### Hair styles: full 111-style catalog (10-row grid)

Separate cell since 111 distinct hair styles don't fit the per-attribute grid layout above.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# =====================================================================
# CONFIGURATION AND FILE PATHS
# =====================================================================
# Base directory where cartoonset 100k PNG folders are kept
# Assuming your loaded metadata DataFrame is named 'df'
# df = pd.read_csv('/kaggle/working/cartoon_attr.csv') 

# Get all unique structural hair values present in your dataset chunk
distinct_hair_vals = sorted(df['hair'].unique())
total_hair_styles = len(distinct_hair_vals)

print(f"Extracting separate hair grid... Found {total_hair_styles} distinct hairstyles.")

if total_hair_styles > 0:
    # We lock down exactly 10 rows as requested
    num_rows = 10
    
    # Calculate how many columns are needed to fit all 111 values (~12 columns)
    num_cols = int(np.ceil(total_hair_styles / num_rows)) 
    
    # Set up the visualization grid layout size dynamically
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 1.8, num_rows * 1.8))
    axes_flat = axes.flatten()
    
    # Process sequentially through every single hair ID index
    for idx, hair_id in enumerate(distinct_hair_vals):
        ax = axes_flat[idx]
        
        # Filter all rows that contain this specific hair type asset
        matching_hair = df[df['hair'] == hair_id]
        
        if len(matching_hair) > 0:
            # Gather exactly 1 random sample row to display diversity
            random_sample = matching_hair.sample(n=1).iloc[0]
            
            # Formulate full image path string safely
            clean_path = str(random_sample['img_path']).replace('/kaggle/working/cartoonset100k/', '')
            img_path = os.path.join(IMAGE_DIR, clean_path)
            
            try:
                img = Image.open(img_path)
                ax.imshow(img)
            except Exception:
                ax.text(0.5, 0.5, "Img\nError", ha='center', va='center', color='red', fontsize=8)
        else:
            ax.text(0.5, 0.5, "Empty\nData", ha='center', va='center', color='gray', fontsize=8)
            
        # Label the individual subplot with its distinct Hair asset value ID
        ax.set_title(f"Hair ID: {hair_id}", fontsize=8, fontweight='bold')
        ax.axis('off')
        
    # Safely disable/hide any excess empty subplots at the tail end of row 10
    for idx in range(total_hair_styles, len(axes_flat)):
        axes_flat[idx].axis('off')
        
    fig.suptitle("Google Cartoon Set: Isolated Hair Styles Catalog (10-Row Grid)", 
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Error: No data available in the 'hair' column to plot.")

## 4. Zero-shot CLIP sanity check

Before fine-tuning anything: does off-the-shelf CLIP even distinguish
cartoon attributes at all? Establishes the "gap" baseline that fine-tuning
(Section 6.3) is measured against.


In [ ]:
!pip install open-clip-torch
import torch
from PIL import Image
import open_clip

model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
model.eval()

In [ ]:
image_paths = df['img_path'][0:100].tolist()
imgs   = [preprocess(Image.open(p)) for p in image_paths]
batch  = torch.stack(imgs)

with torch.no_grad():
    embs = model.encode_image(batch)   # [N, 512]

# cosine similarity between all pairs
embs = embs / embs.norm(dim=-1, keepdim=True)
sim  = embs @ embs.T
print(sim)

In [ ]:
blonde   = df[df['hair_color'] == 0].head(50)
brunette = df[df['hair_color'] == 1].head(50)
sample   = pd.concat([blonde, brunette]).reset_index(drop=True)

labels = torch.tensor([0]*50 + [1]*50)

imgs  = [preprocess(Image.open(p)) for p in sample['img_path'].tolist()]
batch = torch.stack(imgs)

with torch.no_grad():
    embs = model.encode_image(batch)

embs = embs / embs.norm(dim=-1, keepdim=True)
sim  = embs @ embs.T

same_class = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~torch.eye(100).bool()
diff_class = (labels.unsqueeze(0) != labels.unsqueeze(1))

print(f"Avg similarity — same class:  {sim[same_class].mean().item():.4f}")
print(f"Avg similarity — diff class:  {sim[diff_class].mean().item():.4f}")
print(f"Gap: {(sim[same_class].mean() - sim[diff_class].mean()).item():.4f}")

## 5. Generate captions

See `generate_clip_caption` in `src/captions.py` for the full logic
(synonym pools per attribute, 111-style hair geometry map, randomized
ordering and opener phrase -- Appendix B of the thesis).


In [ ]:
df['clip_text_description'] = df.apply(generate_clip_caption, axis=1)
df[['img_path', 'clip_text_description']].head(3)


In [ ]:
import os
import random
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Assuming 'df' is your loaded DataFrame containing the attributes
# and 'generate_clip_caption' is the function we defined earlier

# 1. Grab 3 completely random rows from your dataset
sample_df = df.sample(n=3)

# 2. Setup a clean horizontal plotting grid for the 3 samples
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

# Base directory where cartoonset PNG images are stored
IMAGE_DIR = "/kaggle/working/cartoonset100k/" 

for idx, (row_idx, row) in enumerate(sample_df.iterrows()):
    # Generate the dynamic, shuffled caption for this specific row
    caption_text = generate_clip_caption(row)
    
    # Resolve the physical image path safely
    clean_img_path = str(row['img_path']).replace('/kaggle/working/cartoonset100k/', '')
    img_path = os.path.join(IMAGE_DIR, clean_img_path)
    
    # Plot the image onto the subplot
    try:
        img = Image.open(img_path)
        axes[idx].imshow(img)
    except Exception:
        axes[idx].text(0.5, 0.5, "Image Missing/Error", ha='center', va='center', color='red')
    
    axes[idx].axis('off')
    
    # Format the caption text below the image using wrapping to prevent overlapping text columns
    # We use a custom string join to inject newlines every 6-7 words so it fits perfectly under the picture
    words = caption_text.split()
    wrapped_caption = "\n".join([" ".join(words[i:i+6]) for i in range(0, len(words), 6)])
    
    # Place the text centered below each image block
    axes[idx].text(0.5, -0.15, wrapped_caption, ha='center', va='top', 
                  transform=axes[idx].transAxes, fontsize=10, 
                  fontweight='medium', bbox=dict(boxstyle='round,pad=0.5', facecolor='#f7f7f7', edgecolor='#d3d3d3', alpha=0.9))

plt.tight_layout()
plt.show()

In [ ]:
df.head(1)

## 6. Fine-tune CLIP on (image, caption) pairs

See `CartoonCLIPDataset`, `clip_loss`, `setup_trainable_params`, and
`finetune_clip` in `src/data.py` / `src/training.py`.


In [ ]:
model, tokenizer, preprocess = finetune_clip(
    df,
    save_path       = 'clip_cartoon_finetuned.pt',
    batch_size      = 128,    # increase if you have VRAM
    num_epochs      = 5,
    lr              = 5e-6,
    n_visual_blocks = 2,
    n_text_blocks   = 2,
    device          = device,
)


Verify the fine-tune actually improved attribute separability (should be well above the zero-shot baseline gap measured above).

In [ ]:
# Step 2 — Verify the gap improved (should be >> 0.0072)
check_gap(df, model, preprocess, device=device)

Pre-compute an embedding for every image, so GAN training only ever does an array lookup.

In [ ]:
# Step 3 — Pre-compute all embeddings (run once, ~2 min for 100k on GPU)
embeddings = precompute_text_embeddings(
    df, model, tokenizer,
    save_path  = 'cartoon_clip_embeddings.npy',
    device     = device,
)

In [ ]:
# Step 4 — Save embedding index alongside your df
df['emb_idx'] = range(len(df))
df.head(1)

## 7. Visualize the fine-tuned embedding space (UMAP)

2D UMAP projection of every image's CLIP embedding, colored by attribute.
Tight, well-separated clusters indicate the fine-tuned embedding actually
encodes that attribute (Appendix C of the thesis).


In [ ]:
from umap import umap_ as umap

embeddings = np.load('cartoon_clip_embeddings.npy')   # [N, 512]

reducer = umap.UMAP(n_components=2, random_state=42)
embs_2d = reducer.fit_transform(embeddings)           # [N, 2]

hair_colors_arr = df['hair_color'].values
hair_labels = {
    0: "Very Light Blonde", 1: "Blonde", 2: "Orange", 3: "Red", 4: "Caramel",
    5: "Brunette", 6: "Dark Brown", 7: "Black", 8: "Grey", 9: "Silver"
}
hair_hex_colors = {
    0: "#FFEFA6", 1: "#EED26A", 2: "#FF8C00", 3: "#D32F2F", 4: "#C68E17",
    5: "#8B4513", 6: "#3E2723", 7: "#000000", 8: "#757575", 9: "#BDBDBD",
}

plot_clip_clusters(
    attribute_name="Hair Color",
    target_ids=hair_colors_arr,
    display_labels=hair_labels,
    hex_colors=hair_hex_colors,
    coordinates=embs_2d,
)


Same plot for eye color and skin tone. `eye_color`/`face_color` in `df`
are already the integer attribute IDs from the source CSV (not text), so
they're used directly as `target_ids` -- no extra ID-mapping step needed.


In [ ]:
eye_display_labels = {0: "Brown", 1: "Blue", 2: "Green", 3: "Grey", 4: "Black"}
eye_hex_colors = {
    0: "#593E30", 1: "#2E6F9E", 2: "#3B7A57", 3: "#8A9597", 4: "#1C1C1C",
}

face_display_labels = {
    0: "Ultra-Deep Dark", 1: "Deep Dark", 2: "Dark Brown", 3: "Medium Deep Brown",
    4: "Light Brown", 5: "Tan / Olive", 6: "Medium Warm", 7: "Warm Light",
    8: "Fair", 9: "Very Fair / Pale", 10: "Extremely Pale / Alabaster",
}
face_hex_colors = {
    0: "#22130B", 1: "#331C0E", 2: "#4D2B15", 3: "#663A1C", 4: "#8B5A2B",
    5: "#A07146", 6: "#C69367", 7: "#E6B993", 8: "#F3D1B6", 9: "#FCE5D3", 10: "#FFF3EB",
}

plot_clip_clusters("Eye Color", df['eye_color'].values, eye_display_labels, eye_hex_colors, embs_2d)
plot_clip_clusters("Face Color (Skin Tone)", df['face_color'].values, face_display_labels, face_hex_colors, embs_2d)


In [ ]:
# Step 2 — Verify the gap improved (should be >> 0.0072)
check_gap(df, model, preprocess, device=device)

## 8. Full attribute-gap audit

Runs the same-vs-cross-class similarity check for every conditioning
attribute at once (Table `clip_gap` in the thesis).


In [ ]:
target_attributes = [
    'hair_color',
    'hair',
    'face_color',
    'face_shape',
    'chin_length',
    'eye_color',
    'eyebrow_thickness',
    'facial_hair',
    'glasses',
]

audit_report = audit_all_attribute_gaps(
    df=df,
    model=model,
    preprocess=preprocess,
    attributes=target_attributes,
    device=device,
    samples_per_class=10,  # up to 10 images per unique attribute value
)


## 9. Save outputs

`cartoon_with_embeddings.csv` (captions + attributes), `cartoon_clip_embeddings.npy` (text embeddings), and `clip_cartoon_finetuned.pt` (fine-tuned CLIP weights) are the three files the CartoonSet training notebook needs.

In [ ]:
df.to_csv('cartoon_with_embeddings.csv', index=False)

# To reload the fine-tuned model later (e.g. for text queries at inference):
#
# model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
# model.load_state_dict(torch.load('clip_cartoon_finetuned.pt'))
# model.eval().to(device)
#
# text_emb = F.normalize(
#     model.encode_text(tokenizer(["cartoon with blonde hair"])).to(device),
#     dim=-1
# )
# image = G(noise, text_emb)   # <- inference